<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY4_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from typing import TypedDict, List, Dict, Any
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

In [4]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


In [5]:
class AgentState(TypedDict):
    messages: List[Any]                # Keeps track of the raw chat log
    user_preferences: Dict[str, Any]

In [6]:
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

In [9]:
def chat_node(state: AgentState):
    """
    This node looks at what the user said, reviews its memory scratchpad,
    and updates both the conversation history and its understanding of the user.
    """

    system_prompt = SystemMessage(content=(
        "You are a friendly, stateful chat assistant. "
        "You must pay close attention to the known facts about the user listed below. "
        "Use them to personalize your responses naturally.\n\n"
        f"--- KNOWN USER PREFERENCES (SCRATCHPAD) ---\n{state['user_preferences']}"
    ))

    full_history = [system_prompt] + state["messages"]

    response = model.invoke(full_history)

    last_user_msg = state["messages"][-1].content
    memory_extractor_prompt = (
        f"Analyze this phrase: '{last_user_msg}'. "
        f"Current scratchpad: {state['user_preferences']}. "
        "If the user shared a new preference (like a name, favorite thing, or constraint), "
        "output a clean, updated Python dictionary containing ALL rules. Otherwise, return the current one exactly. "
        "Output ONLY the dictionary raw text, nothing else."
    )

    try:
        extracted_memory_text = model.invoke(memory_extractor_prompt).content
        import ast
        updated_preferences = ast.literal_eval(extracted_memory_text.strip())
    except Exception:

        updated_preferences = state["user_preferences"]

    return {
        "messages": [response],
        "user_preferences": updated_preferences
    }

In [10]:
builder = StateGraph(AgentState)

builder.add_node("chat_assistant", chat_node)

builder.add_edge(START, "chat_assistant")
builder.add_edge("chat_assistant", END)

compiled_agent = builder.compile()

In [15]:
current_session_state = {
    "messages": [],
    "user_preferences": {}
}

In [16]:
def talk_to_agent(user_text: str):
    global current_session_state

    current_session_state["messages"].append(HumanMessage(content=user_text))


    print(f"\nUser: {user_text}")
    current_session_state = compiled_agent.invoke(current_session_state)

    latest_reply = current_session_state["messages"][-1].content
    print(f"Agent: {latest_reply}")
    print(f"[LOCAL SCRATCHPAD STATE]: {current_session_state['user_preferences']}")

In [17]:
talk_to_agent("Hey there! My name is Alex and I absolutely love dark roast coffee.")

talk_to_agent("What is the capital of France?")

talk_to_agent("What should I make myself to drink with breakfast tomorrow?")


User: Hey there! My name is Alex and I absolutely love dark roast coffee.
Agent: Nice to meet you, Alex. I've taken note that you're a fan of dark roast coffee - I'll keep that in mind for our conversation. What's your favorite way to enjoy your dark roast, black or with some cream and sugar?
[LOCAL SCRATCHPAD STATE]: {'name': 'Alex', 'preferences': {'coffee': 'dark roast'}}

User: What is the capital of France?
Agent: Alex, the capital of France is Paris. By the way, I've heard that the French are known for their exquisite coffee culture, and dark roast is a popular choice among the locals. Have you ever tried a French dark roast coffee?
[LOCAL SCRATCHPAD STATE]: {'name': 'Alex', 'preferences': {'coffee': 'dark roast'}}

User: What should I make myself to drink with breakfast tomorrow?
Agent: Considering your love for dark roast coffee, I think a freshly brewed cup of dark roast would be the perfect accompaniment to your breakfast tomorrow, Alex. Would you like some suggestions for a